In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0", "fqdn": "1.6.0", "h11": "0.16.0

# Разбор A. Проверяем запись истрадефиллина

Перед запуском прочитайте [постановку](../../../../../../theory/modules/M03/lessons/L01/01_problem.md), [модель](../../../../../../theory/modules/M03/lessons/L01/02_model.md) и [процедуру](../../../../../../theory/modules/M03/lessons/L01/03_workflow.md). Здесь повторим решение в исполняемых шагах. Отчёт строим из наблюдаемых данных, а не из имени молекулы.

In [2]:
from pathlib import Path
import sys, json, math
import matplotlib.pyplot as plt
from IPython.display import display, Image

# Path.cwd() — текущая папка; parents позволяет найти корень при запуске из папки Notebook.
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import compound, workspace, write_json
from rdkit import Chem,rdBase
from rdkit.Chem import Descriptors,rdMolDescriptors
OUT=workspace('M03-L01','case')
record=compound('istradefylline')
display(record)


{'name': 'istradefylline',
 'cid': 5311037,
 'smiles': 'CCN1C2=C(C(=O)N(C1=O)CC)N(C(=N2)/C=C/C3=CC(=C(C=C3)OC)OC)C',
 'formula': 'C20H24N4O4',
 'molecular_weight_g_mol': 384.4,
 'pubchem_XLogP': 2.5,
 'pubchem_TPSA_A2': 76.9,
 'source_url': 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/istradefylline/property/MolecularFormula,MolecularWeight,CanonicalSMILES,IsomericSMILES,XLogP,TPSA/JSON',
 'source_sha256': 'e74a82530b8e7096e37c1f893b42a60f454c86a805a4df59808961665ac2e80b',
 'property_kind': 'database representation/computed descriptor; not an assay or measured solubility'}

Словарь `record` хранит поля одной записи. Обращение по ключу `record['smiles']` выбирает конкретное представление. Передадим его химическому парсеру. Успешный синтаксический разбор не подтверждает, что реальный образец имеет этот состав.

In [3]:
raw_smiles=record['smiles']
mol=Chem.MolFromSmiles(raw_smiles)
if mol is None:
    raise ValueError('Источник не удалось прочитать как молекулярный граф')

# isomericSmiles=True сохраняет доступную информацию о конфигурации.
isomeric=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True)
connectivity=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=False)
display({'raw':raw_smiles,'canonical_isomeric':isomeric,'connectivity_only':connectivity})


{'raw': 'CCN1C2=C(C(=O)N(C1=O)CC)N(C(=N2)/C=C/C3=CC(=C(C=C3)OC)OC)C',
 'canonical_isomeric': 'CCn1c(=O)c2c(nc(/C=C/c3ccc(OC)c(OC)c3)n2C)n(CC)c1=O',
 'connectivity_only': 'CCn1c(=O)c2c(nc(C=Cc3ccc(OC)c(OC)c3)n2C)n(CC)c1=O'}

До следующей ячейки назовите, какой символ в строке может относиться к геометрии двойной связи. Затем проверим признаки самого графа. `for` перебирает связи; список сохраняет наблюдения, которые позднее можно проверить.

In [4]:
bond_stereo=[]
for bond in mol.GetBonds():
    if bond.GetStereo()!=Chem.BondStereo.STEREONONE:
        bond_stereo.append({'atom_indices':[bond.GetBeginAtomIdx(),bond.GetEndAtomIdx()],
                            'stereo':str(bond.GetStereo())})

# Индекс атома — адрес в этом графе, не номер атома по химической номенклатуре.
summary={'cid':record['cid'],'formula':rdMolDescriptors.CalcMolFormula(mol),
         'molecular_weight_g_mol':Descriptors.MolWt(mol),
         'heavy_atoms':mol.GetNumHeavyAtoms(),'atoms_with_explicit_H':Chem.AddHs(mol).GetNumAtoms(),
         'double_bond_stereo':bond_stereo,'rdkit':rdBase.rdkitVersion}
display(summary)


{'cid': 5311037,
 'formula': 'C20H24N4O4',
 'molecular_weight_g_mol': 384.43600000000015,
 'heavy_atoms': 28,
 'atoms_with_explicit_H': 52,
 'double_bond_stereo': [{'atom_indices': [15, 16], 'stereo': 'STEREOE'}],
 'rdkit': '2026.09.1'}

Теперь проверим ловушку: упрощённая запись может утратить конфигурацию, сохранив массу и формулу. Поэтому совпадение двух чисел не заменяет выбранную политику идентичности.

In [5]:
simplified=Chem.MolFromSmiles(connectivity)
assert simplified is not None
assert rdMolDescriptors.CalcMolFormula(simplified)==summary['formula']
assert math.isclose(Descriptors.MolWt(simplified),summary['molecular_weight_g_mol'],abs_tol=1e-9)
assert Chem.MolToSmiles(simplified,isomericSmiles=True)!=isomeric
write_json(OUT/'identity_passport.json',{'source':record,'computed':summary,
          'canonical_isomeric':isomeric,'connectivity_only':connectivity,
          'conclusion':'Масса и формула не доказывают сохранение всей стереохимии',
          'next_check':'Уточнить соответствие экспериментального образца выбранной конфигурации'})
print(OUT/'identity_passport.json')


C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\practicuum\evidence\20261119-main-unification\notebooks\workspace\M03-L01\case\identity_passport.json


Мы обнаружили потерю информации при определённом преобразовании. Не установили активность, чистоту или свойства неизвестного образца. В досье переносится паспорт и ограничение. На следующем уроке проверим, что меняется при работе с солевой и лактонной формами.